In [5]:
# ==============================================================================
# 2026改修版: GPR_Linear (Jsc, 旧ロジック互換の特徴量選択 + 外れ値解析)
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
  library(stringr)
})

set.seed(42)

# -------------------------------
# パスと対象ファイル
# -------------------------------
base_path   <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")   # ★旧コードと同じ
file_names  <- c("m_all_OH_rebuilt.csv")         # Jsc 補完モデルだけ回す場合

# 出力ルート（outlier 配下）
out_root   <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
model_name <- "GPR_Linear"

# -------------------------------
# 除外変数設定（旧コードと同じ）
# -------------------------------
inappropriate_vars <- c(
  "Jsccal", "JscJsccal", "PCEaTA", "PCEaTAFinal", "EQEABEST", "Rshthin", "Dresistance",
  "mobilityeblendOFET", "mobilityep1MOFET", "mobilityhblendSCLC", "mobilityhnMSCLC",
  "mobilityhp1MSCLC", "IonIoffF", "DRMS", "Var.1"
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPER 関数
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  as.numeric(r^2)
}

calc_gpr_linear_importance <- function(model, data_scaled, target, features) {
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2   <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled
    temp[[col]] <- sample(temp[[col]])
    shuffled_pred <- tryCatch(
      predict(model, temp[, features, drop = FALSE]),
      error = function(e) NA
    )
    if (all(is.na(shuffled_pred))) {
      out_imps[[col]] <- 0
    } else {
      new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
      out_imps[[col]] <- max(0, base_r2 - new_r2)
    }
  }
  data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
}

flag_high_error <- function(df_pred, quantile_prob = 0.9) {
  thr <- stats::quantile(df_pred$AbsError, quantile_prob, na.rm = TRUE)
  df_pred$HighErrorFlag      <- df_pred$AbsError >= thr
  df_pred$HighErrorThreshold <- thr
  df_pred
}

# -------------------------------
# MAIN
# -------------------------------
summary_list    <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) {
    cat("[WARN] File not found:", filepath, "\n")
    next
  }
  
  df_raw <- tryCatch(
    read.csv(filepath, row.names = 1, check.names = FALSE),
    error = function(e) NULL
  )
  if (is.null(df_raw)) {
    cat("[WARN] Failed to read:", filepath, "\n")
    next
  }
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL
  
  # ★旧コードと同じく、target_vars を総当たり
  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) next
    if (target != "Jsc") next   # 必要なら Jsc だけに絞る
    
    # 出力ディレクトリ
    tag     <- paste0(tools::file_path_sans_ext(file), "_", target)
    run_dir <- file.path(out_root, paste0(model_name, "_", tag))
    if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)
    
    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) {
      cat("[WARN] Too few rows:", nrow(df_temp), "for", tag, "\n")
      next
    }
    
    # ----------------------------
    # ★特徴量選択：旧コードと同じロジック
    # ----------------------------
    # 旧: features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    if (length(physical_exclude_patterns) > 0) {
      features <- features[!grepl(paste(physical_exclude_patterns, collapse = "|"), features)]
    }
    
    # ゼロ分散除外
    vars     <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]
    
    # 多重共線性対策
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) {
      cat("[WARN] No usable features for", tag, "\n")
      next
    }
    
    # ----------------------------
    # スケーリング [-1,1]
    # ----------------------------
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1
    df_scaled$SampleID <- rownames(df_scaled)
    
    # ----------------------------
    # CV10 設定 & 学習
    # ----------------------------
    train_ctrl <- trainControl(
      method = "cv",
      number = 10,
      savePredictions = "final",
      returnResamp   = "all"
    )
    
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE],
        y = df_scaled[[target]],
        method    = "gaussprLinear",
        trControl = train_ctrl,
        tuneGrid  = NULL,
        metric    = "RMSE"
      ),
      error = function(e) {
        cat("  [ERROR] GPR_Linear failed:", e$message, "\n")
        return(NULL)
      }
    )
    if (is.null(model)) next
    
    # 最適（唯一）条件の表示
    cat(sprintf(
      "[BEST] %s - %s | gaussprLinear | RMSE=%.4f, Rsq=%.4f\n",
      file, target, model$results$RMSE, model$results$Rsquared
    ))
    
    # ----------------------------
    # OOF + foldID + 誤差
    # ----------------------------
    pred_oof <- model$pred %>%
      mutate(
        SampleID = rownames(df_scaled)[rowIndex],
        foldID   = Resample,
        Type     = "CV10_OOF",
        Observed = obs,
        Predicted = pred,
        Error    = Predicted - Observed,
        AbsError = abs(Error)
      ) %>%
      dplyr::select(SampleID, foldID, Observed, Predicted, Error, AbsError, Type)
    
    pred_oof <- flag_high_error(pred_oof, quantile_prob = 0.9)
    
    # 特徴量結合
    feat_df <- df_scaled[, c("SampleID", features), drop = FALSE]
    pred_with_feat <- pred_oof %>% left_join(feat_df, by = "SampleID")
    
    # ----------------------------
    # 保存
    # ----------------------------
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))
    saveRDS(pp,    file.path(run_dir, paste0(tag, "_preprocess.rds")))
    
    write.csv(pred_oof,
              file.path(run_dir, paste0(tag, "_pred_OOF_withError.csv")),
              row.names = FALSE)
    write.csv(pred_with_feat,
              file.path(run_dir, paste0(tag, "_pred_OOF_withError_andFeatures.csv")),
              row.names = FALSE)
    write.csv(pred_with_feat[pred_with_feat$HighErrorFlag, ],
              file.path(run_dir, paste0(tag, "_highErrorSamples.csv")),
              row.names = FALSE)
    
    # 重要度
    imp_df <- calc_gpr_linear_importance(model, df_scaled, target, features)
    imp_df$File   <- file
    imp_df$Target <- target
    imp_df$Model  <- model_name
    importance_list[[length(importance_list) + 1]] <- imp_df
    
    # サマリー
    cv10_r2   <- safe_r2(pred_oof$Observed, pred_oof$Predicted)
    cv10_rmse <- rmse(pred_oof$Observed, pred_oof$Predicted)
    
    summary_list[[length(summary_list) + 1]] <- data.frame(
      File        = file,
      Target      = target,
      Model       = model_name,
      CV10_R2     = cv10_r2,
      CV10_RMSE   = cv10_rmse,
      n_samples   = nrow(df_scaled),
      n_features  = length(features),
      Params      = "(none)",
      HighErrorThreshold = unique(pred_oof$HighErrorThreshold)
    )
    
    cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f | n=%d | p=%d\n",
                file, target, cv10_r2, nrow(df_scaled), length(features)))
  }
}

if (length(summary_list) > 0) {
  all_summary <- do.call(rbind, summary_list)
  write.csv(
    all_summary,
    file.path(out_root, paste0("all_summary_CV10_", model_name, "_oldFeatLogic.csv")),
    row.names = FALSE
  )
}
if (length(importance_list) > 0) {
  all_imp <- do.call(rbind, importance_list)
  write.csv(
    all_imp,
    file.path(out_root, paste0("all_importance_", model_name, "_oldFeatLogic.csv")),
    row.names = FALSE
  )
}

cat("\n[DONE] GPR_Linear (old feature-selection logic) completed.\n")


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."


[BEST] m_all_OH_rebuilt.csv - Jsc | gaussprLinear | RMSE=2.4056, Rsq=0.7440
  Finished: m_all_OH_rebuilt.csv - Jsc | CV10_R2: 0.741 | n=1045 | p=427

[DONE] GPR_Linear (old feature-selection logic) completed.


In [3]:
# # ==============================================================================
# # 2026改修版: GPR_Linear + 外れ値解析用フル版
# # - 10-fold CV + foldID 付き OOF 予測
# # - 誤差 (Error, AbsError)、高誤差フラグ
# # - 特徴量テーブル（S / 非S 判別フラグ付き）出力
# # - Permutation Importance (R2 drop)
# # ==============================================================================

# suppressPackageStartupMessages({
#   library(caret)
#   library(dplyr)
#   library(Metrics)
#   library(kernlab)
#   library(stringr)
# })

# set.seed(42)

# # -------------------------------
# # パスと対象ファイル
# # -------------------------------
# base_path   <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
# target_vars <- c("Jsc")
# file_names  <- c("m_all_OH_rebuilt.csv")

# # 出力先（SVM 版と同様に outlier 配下）
# today    <- format(Sys.Date(), "%Y%m%d")
# out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
# model_name <- "GPR_Linear"

# # -------------------------------
# # 除外変数設定
# # -------------------------------
# inappropriate_vars <- c(
#   'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
#   'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
#   'IonIoffF', 'DRMS', 'Var.1'
# )
# physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# # -------------------------------
# # HELPERS
# # -------------------------------
# safe_r2 <- function(y, p) {
#   r <- suppressWarnings(cor(y, p))
#   if (is.na(r)) return(NA_real_)
#   as.numeric(r^2)
# }

# # Permutation Importance 関数（線形 GPR 用）
# calc_gpr_linear_importance <- function(model, data_scaled, target, features) {
#   base_pred <- predict(model, data_scaled[, features, drop = FALSE])
#   base_r2   <- safe_r2(data_scaled[[target]], base_pred)
  
#   out_imps <- list()
#   for (col in features) {
#     temp <- data_scaled
#     temp[[col]] <- sample(temp[[col]])  # シャッフル
#     shuffled_pred <- tryCatch(
#       predict(model, temp[, features, drop = FALSE]),
#       error = function(e) NA
#     )
    
#     if (all(is.na(shuffled_pred))) {
#       out_imps[[col]] <- 0
#     } else {
#       new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
#       out_imps[[col]] <- max(0, base_r2 - new_r2)
#     }
#   }
#   data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
# }

# # 高誤差サンプルを定義する関数（上位10%を S にする例）
# flag_high_error <- function(df_pred, quantile_prob = 0.9) {
#   thr <- stats::quantile(df_pred$AbsError, quantile_prob, na.rm = TRUE)
#   df_pred$HighErrorFlag       <- df_pred$AbsError >= thr
#   df_pred$HighErrorThreshold  <- thr
#   df_pred
# }

# # -------------------------------
# # MAIN LOOP
# # -------------------------------
# summary_list    <- list()
# importance_list <- list()

# for (file in file_names) {
#   filepath <- file.path(base_path, file)
#   if (!file.exists(filepath)) {
#     cat("[WARN] File not found:", filepath, "\n")
#     next
#   }
  
#   df_raw <- tryCatch(
#     read.csv(filepath, row.names = 1, check.names = FALSE),
#     error = function(e) NULL
#   )
#   if (is.null(df_raw)) {
#     cat("[WARN] Failed to read:", filepath, "\n")
#     next
#   }
#   if ("X" %in% colnames(df_raw)) df_raw$X <- NULL
  
#   for (target in target_vars) {
#     if (!target %in% colnames(df_raw)) {
#       cat("[WARN] Target not found:", target, "in", file, "\n")
#       next
#     }
    
#     # 出力ディレクトリ（ファイル x ターゲット）
#     tag     <- paste0(tools::file_path_sans_ext(file), "_", target)
#     run_dir <- file.path(out_root, paste0(model_name, "_", tag))
#     if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)
    
#     # ----------------------------
#     # クレンジング & 変数選択
#     # ----------------------------
#     df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
#     if (nrow(df_temp) < 20) {
#       cat("[WARN] Too few rows after na.omit:", nrow(df_temp), "for", tag, "\n")
#       next
#     }
    
#     features <- setdiff(colnames(df_temp), target)
#     features <- setdiff(features, inappropriate_vars)
#     if (length(physical_exclude_patterns) > 0) {
#       features <- features[!grepl(paste(physical_exclude_patterns, collapse = "|"), features)]
#     }
    
#     # ゼロ分散除外
#     vars     <- sapply(df_temp[, features, drop = FALSE], var)
#     features <- names(vars)[vars > 0 & !is.na(vars)]
    
#     # 多重共線性対策 (Corr > 0.99999)
#     if (length(features) > 1) {
#       cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
#       cor_mat[is.na(cor_mat)] <- 0
#       high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
#       if (length(high_corr) > 0) features <- features[-high_corr]
#     }
#     if (length(features) == 0) {
#       cat("[WARN] No usable features for", tag, "\n")
#       next
#     }
    
#     # ----------------------------
#     # スケーリング [-1, 1]
#     # ----------------------------
#     pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
#     df_scaled <- df_temp
#     df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1
#     df_scaled$SampleID <- rownames(df_scaled)
    
#     # ----------------------------
#     # 10-fold CV 設定 & 学習
#     # ----------------------------
#     train_ctrl <- trainControl(
#       method = "cv",
#       number = 10,
#       savePredictions = "final",
#       returnResamp   = "all"
#     )
    
#     model <- tryCatch(
#       train(
#         x = df_scaled[, features, drop = FALSE],
#         y = df_scaled[[target]],
#         method    = "gaussprLinear",
#         trControl = train_ctrl,
#         tuneGrid  = NULL,
#         metric    = "RMSE"
#       ),
#       error = function(e) {
#         cat("  [ERROR] GPR_Linear failed:", e$message, "\n")
#         return(NULL)
#       }
#     )
#     if (is.null(model)) next
#     cat(sprintf(
#       "[BEST] %s - %s | gaussprLinear (no tuning) | RMSE=%.4f, Rsq=%.4f\n",
#       file, target,
#       model$results$RMSE,
#       model$results$Rsquared
#     ))
#     ## ▼ チューニング結果保存 ▼
#     tune_tbl <- model$results
#     tune_tbl$File   <- file
#     tune_tbl$Target <- target
#     tune_tbl$Model  <- model_name
#     write.csv(
#       tune_tbl,
#       file.path(run_dir, paste0(tag, "_tuning_results.csv")),
#       row.names = FALSE
#     )
#     # GPR_Linear はチューニングパラメータなしだが、同じ形式で best を保存
#     best_row <- tune_tbl
#     write.csv(
#       best_row,
#       file.path(run_dir, paste0(tag, "_best_params.csv")),
#       row.names = FALSE
#     )
#     ## ▲ ここまで ▲
      
    
#     # ----------------------------
#     # OOF 予測 + foldID + 誤差
#     # ----------------------------
#     pred_oof <- model$pred %>%
#       mutate(
#         SampleID = rownames(df_scaled)[rowIndex],
#         foldID   = Resample,
#         Type     = "CV10_OOF",
#         Observed = obs,
#         Predicted = pred,
#         Error    = Predicted - Observed,
#         AbsError = abs(Error)
#       ) %>%
#       dplyr::select(SampleID, foldID, Observed, Predicted, Error, AbsError, Type)
    
#     # 高誤差フラグ付与（上位10%）
#     pred_oof <- flag_high_error(pred_oof, quantile_prob = 0.9)
    
#     # ----------------------------
#     # 特徴量テーブルとの結合（解析用）
#     # ----------------------------
#     feat_df <- df_scaled[, c("SampleID", features), drop = FALSE]
#     pred_with_feat <- pred_oof %>%
#       left_join(feat_df, by = "SampleID")
    
#     # ----------------------------
#     # モデル保存
#     # ----------------------------
#     saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))
#     saveRDS(pp,    file.path(run_dir, paste0(tag, "_preprocess.rds")))
    
#     # ----------------------------
#     # CSV 出力
#     # ----------------------------
#     # 1) OOF 予測（誤差・foldID 付き）
#     write.csv(
#       pred_oof,
#       file.path(run_dir, paste0(tag, "_pred_OOF_withError.csv")),
#       row.names = FALSE
#     )
    
#     # 2) 特徴量付きテーブル（S 判別用）
#     write.csv(
#       pred_with_feat,
#       file.path(run_dir, paste0(tag, "_pred_OOF_withError_andFeatures.csv")),
#       row.names = FALSE
#     )
    
#     # 3) 高誤差サンプルのみ
#     high_err_df <- pred_with_feat[pred_with_feat$HighErrorFlag, ]
#     write.csv(
#       high_err_df,
#       file.path(run_dir, paste0(tag, "_highErrorSamples.csv")),
#       row.names = FALSE
#     )
    
#     # ----------------------------
#     # Permutation Importance
#     # ----------------------------
#     imp_df <- calc_gpr_linear_importance(model, df_scaled, target, features)
#     imp_df$File   <- file
#     imp_df$Target <- target
#     imp_df$Model  <- model_name
#     importance_list[[length(importance_list) + 1]] <- imp_df
    
#     # ----------------------------
#     # サマリー集計
#     # ----------------------------
#     cv10_r2   <- safe_r2(pred_oof$Observed, pred_oof$Predicted)
#     cv10_rmse <- rmse(pred_oof$Observed, pred_oof$Predicted)
    
#     summary_list[[length(summary_list) + 1]] <- data.frame(
#       File        = file,
#       Target      = target,
#       Model       = model_name,
#       CV10_R2     = cv10_r2,
#       CV10_RMSE   = cv10_rmse,
#       n_samples   = nrow(df_scaled),
#       n_features  = length(features),
#       Params      = "(none)",
#       HighErrorThreshold = unique(pred_oof$HighErrorThreshold)
#     )
    
#     cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f | n=%d | p=%d\n",
#                 file, target, cv10_r2, nrow(df_scaled), length(features)))
#   }
# }

# # -------------------------------
# # グローバル CSV 出力
# # -------------------------------
# if (length(summary_list) > 0) {
#   all_summary <- do.call(rbind, summary_list)
#   write.csv(
#     all_summary,
#     file.path(out_root, paste0("all_summary_CV10_", model_name, ".csv")),
#     row.names = FALSE
#   )
# }

# if (length(importance_list) > 0) {
#   all_imp <- do.call(rbind, importance_list)
#   write.csv(
#     all_imp,
#     file.path(out_root, paste0("all_importance_", model_name, ".csv")),
#     row.names = FALSE
#   )
# }

# cat("\n[DONE] GPR_Linear Outlier-aware Analysis completed.\n")


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."


[BEST] m_all_OH_rebuilt.csv - Jsc | gaussprLinear (no tuning) | RMSE=1.0225, Rsq=0.9531
  Finished: m_all_OH_rebuilt.csv - Jsc | CV10_R2: 0.953 | n=1045 | p=430

[DONE] GPR_Linear Outlier-aware Analysis completed.
